# LightGBM Model

In [18]:
# Imports
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

In [19]:
# Load datasets 
train_df = pd.read_csv("/Users/mateopozoruiz/Projects/Housing-Pricing-End-to-End-ML/data/processed/feature_engineered_train_data.csv")
test_df = pd.read_csv("/Users/mateopozoruiz/Projects/Housing-Pricing-End-to-End-ML/data/processed/feature_engineered_test_data.csv")

In [20]:
# Convert dataframes into LightGBM dataset objects
train_data = lgb.Dataset(train_df.drop(columns=["price"]), label=train_df["price"])
test_data = lgb.Dataset(test_df.drop(columns=["price"]), label=test_df["price"], reference=train_data)

## Train and Evaluate the model

In [43]:
params = {
    'objective': 'regression',
    'metric': 'mae,rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.15,
    'num_leaves': 31,
}

num_round = 100

bst = lgb.train(params, train_data, num_round, valid_sets=[test_data], callbacks=[lgb.early_stopping(stopping_rounds=5)])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012835 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7639
[LightGBM] [Info] Number of data points in the train set: 576815, number of used features: 39
[LightGBM] [Info] Start training from score 340113.274741
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's l1: 33980.8	valid_0's rmse: 72038.8


In [47]:
# Calculate metrics 

y_pred = bst.predict(test_df.drop(columns=["price"]), num_iteration=bst.best_iteration)

r2 = r2_score(test_df["price"], y_pred)
rmse = np.sqrt(mean_squared_error(test_df["price"], y_pred))
mae = mean_absolute_error(test_df["price"], y_pred)

print(f"MAE:{mae:,.2f}")
print(f"RMSE:{rmse:,.2f}")
print(f"R²:{r2:,.4f}")

MAE:33,980.77
RMSE:72,038.82
R²:0.9599
